In [3]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import Accuracy
import pandas as pd
from tensorflow.keras.utils import plot_model

In [ ]:
#!pip install numpy==1.23.5


In [7]:
import pandas as pd
import numpy as np
from sklearn.utils import resample

def balance_multilabel_df(df, label_columns, method='oversample'):
    # Create a new dataframe to hold the balanced data
    df_balanced = pd.DataFrame(columns=df.columns)
    
    if method == 'oversample':
        # Calculate the maximum count of any label to determine the target count for oversampling
        target_count = df[label_columns].sum().max()
    elif method == 'undersample':
        # Calculate the minimum count of any label to determine the target count for undersampling
        target_count = df[label_columns].sum().min()
    else:
        raise ValueError("Method must be 'oversample' or 'undersample'")

    # Iterate over each label to balance the dataset
    for label in label_columns:
        # Separate the instances with and without the current label
        df_with_label = df[df[label] == 1]
        df_without_label = df[df[label] == 0]
        
        if method == 'oversample':
            # Oversample the instances with the label to match the target count
            df_with_label_resampled = resample(df_with_label,
                                               replace=True,  # Sample with replacement
                                               n_samples=int(target_count),  # Target count
                                               random_state=42)
            # Combine the resampled instances with the current balanced dataframe
            df_balanced = pd.concat([df_balanced, df_with_label_resampled])
        elif method == 'undersample':
            # Undersample the instances without the label to match the target count, ensuring we do not exceed available instances
            df_without_label_resampled = resample(df_without_label,
                                                  replace=False,  # Sample without replacement
                                                  n_samples=min(len(df_without_label), int(target_count)),  # Target count
                                                  random_state=42)
            # Combine the resampled instances with the current balanced dataframe
            df_balanced = pd.concat([df_balanced, df_without_label_resampled])
    
    # Drop duplicates that might have been added multiple times during oversampling
    df_balanced = df_balanced.drop_duplicates().reset_index(drop=True)

    return df_balanced

In [20]:
print(np.__version__)

1.23.5


In [9]:
import os
import pandas as pd

# Define data directory and file paths
data_dir = "../data"
file_paths = {
    "graphics_real": os.path.join(data_dir, "graphicsReal.csv"),
    "graphics_not_real": os.path.join(data_dir, "graphicsNotReal.csv"),
    "bullets_real": os.path.join(data_dir, "bulletsReal.csv"),
    "bullets_not": os.path.join(data_dir, "NoBullets.csv"),
    "readable": os.path.join(data_dir, "readable.csv"),
    "not_readable": os.path.join(data_dir, "NoReadable.csv"),
    "boring": os.path.join(data_dir, "boring.csv"),
    "interesting": os.path.join(data_dir, "interesting.csv"),
    "agenda_yes": os.path.join(data_dir, "agendaYes.csv"),
    "text_size_ok": os.path.join(data_dir, "sizeOfTextOk.csv"),
    "text_size_not_ok": os.path.join(data_dir, "sizeOfTextNotOk.csv"),
    "positioning_ok": os.path.join(data_dir, "positioningOk.csv"),
    "positioning_not_ok": os.path.join(data_dir, "positioningNotOk.csv"),
    "pictures": os.path.join(data_dir, "pictures.csv"),
    "no_pictures": os.path.join(data_dir, "noPictures.csv"),
    "tables": os.path.join(data_dir, "tables.csv"),
    "infographics": os.path.join(data_dir, "infographics.csv"),
    "no_infographics": os.path.join(data_dir, "noInfographics.csv")
}

# Safe CSV loading function
import csv
data_dir = "../data/"
def safe_read_csv(path, name):
    if os.path.exists(path):
        try:
            # Auto-detect delimiter
            with open(path, 'r', encoding='utf-8') as f:
                sample = f.read(2048)
                dialect = csv.Sniffer().sniff(sample, delimiters=[',', '\t'])
                delimiter = dialect.delimiter
            
            df = pd.read_csv(path, delimiter=delimiter)
            df.columns = df.columns.str.strip()

            # Normalize img_path column if present
            if 'img_path' in df.columns:
                df['img_path'] = (
                    df['img_path']
                    .astype(str)
                    .str.replace('\\', '/', regex=False)
                    .str.replace('D:/WorkspaceAI', data_dir, regex=False)
                )

            return df
        except Exception as e:
            print(f"❌ Failed to read {name}: {e}")
            return None
    else:
        print(f"⚠️ File not found: {name} -> {path}")
        return None



# Load all CSVs using the safer structure
df_graphics = safe_read_csv(file_paths["graphics_real"], "graphics_real")
df_graphics_not = safe_read_csv(file_paths["graphics_not_real"], "graphics_not_real")
df_bullets = safe_read_csv(file_paths["bullets_real"], "bullets_real")
df_no_bullets = safe_read_csv(file_paths["bullets_not"], "bullets_not")
df_readable = safe_read_csv(file_paths["readable"], "readable")
df_no_readable = safe_read_csv(file_paths["not_readable"], "not_readable")
df_boring = safe_read_csv(file_paths["boring"], "boring")
df_interesting = safe_read_csv(file_paths["interesting"], "interesting")
df_agenda_yes = safe_read_csv(file_paths["agenda_yes"], "agenda_yes")
df_text_size_ok = safe_read_csv(file_paths["text_size_ok"], "text_size_ok")
df_text_size_not_ok = safe_read_csv(file_paths["text_size_not_ok"], "text_size_not_ok")
df_positioning_ok = safe_read_csv(file_paths["positioning_ok"], "positioning_ok")
df_positioning_not_ok = safe_read_csv(file_paths["positioning_not_ok"], "positioning_not_ok")
df_pictures = safe_read_csv(file_paths["pictures"], "pictures")
df_no_pictures = safe_read_csv(file_paths["no_pictures"], "no_pictures")
df_tables = safe_read_csv(file_paths["tables"], "tables")
df_infographics = safe_read_csv(file_paths["infographics"], "infographics")
df_no_infographics = safe_read_csv(file_paths["no_infographics"], "no_infographics")

standard_columns = [ 'img_path', 'bullets', 'interesting', 'readable', 'agenda', 'size_of_text', 'tables', 'positioning', 'pictures', 'infographics', 'graphics']

# Function to standardize columns
def standardize_columns(df, df_name, columns):
    # Add missing columns with NaN values
    for col in columns:
        if col not in df.columns:
           raise Exception("Missing column "+ str(col)+" from dataset"+ str(df_name))
    # Ensure the correct order of columns
    df = df[columns]
    return df

    
data = [df_graphics, df_graphics_not, df_bullets, df_no_bullets, df_readable, df_no_readable,df_boring,df_interesting,
                 df_agenda_yes,df_text_size_ok,df_text_size_not_ok,df_positioning_ok,df_positioning_not_ok,df_pictures,df_no_pictures,
                 df_tables,df_infographics,df_no_infographics]


In [14]:
!pip install Pillow
!pip show Pillow


   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   ------ --------------------------------- 1.0/7.0 MB 5.6 MB/s eta 0:00:02
   ------------- -------------------------- 2.4/7.0 MB 5.6 MB/s eta 0:00:01
   --------------------- ------------------ 3.7/7.0 MB 5.7 MB/s eta 0:00:01
   --------------------------- ------------ 4.7/7.0 MB 5.7 MB/s eta 0:00:01
   ---------------------------------- ----- 6.0/7.0 MB 5.8 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 5.6 MB/s eta 0:00:00
Name: pillow
Version: 11.3.0
Summary: Python Imaging Library (Fork)
Home-page: https://python-pillow.github.io
Author: 
Author-email: "Jeffrey A. Clark" <aclark@aclark.net>
License-Expression: MIT-CMU
Location: c:\users\maria\anaconda3\envs\tf_gpu\lib\site-packages
Requires: 
Required-by: 


In [24]:
# import pandas as pd

# # Load the dataframes
# df_graphics = pd.read_csv('graphicsReal.csv')
# df_graphics_not = pd.read_csv('graphicsNotReal.csv')
# df_bullets = pd.read_csv('bulletsReal.csv')
# df_no_bullets = pd.read_csv('NoBullets.csv')
# df_readable = pd.read_csv('readable.csv')
# df_no_readable = pd.read_csv('NoReadable.csv')
# df_boring = pd.read_csv('boring.csv', delimiter='\t')
# df_interesting = pd.read_csv('interesting.csv', delimiter='\t')
# df_agenda_yes = pd.read_csv('agendaYes.csv', delimiter='\t')
# df_text_size_ok = pd.read_csv('sizeOfTextOk.csv', delimiter='\t')
# df_text_size_not_ok = pd.read_csv('sizeOfTextNotOk.csv', delimiter='\t')
# df_positioning_ok = pd.read_csv('positioningOk.csv', delimiter='\t')
# df_positioning_not_ok = pd.read_csv('positioningNotOk.csv', delimiter='\t')
# df_pictures = pd.read_csv('pictures.csv', delimiter='\t')
# df_no_pictures = pd.read_csv('noPictures.csv', delimiter='\t')
# df_tables = pd.read_csv('tables.csv', delimiter='\t')
# df_infographics = pd.read_csv('infographics.csv', delimiter='\t')
# df_no_infographics = pd.read_csv('noInfographics.csv', delimiter='\t')

# # Create a dictionary of dataframes
# dataframes = {
#     'df_graphics': df_graphics,
#     'df_graphics_not': df_graphics_not,
#     'df_bullets': df_bullets,
#     'df_no_bullets': df_no_bullets,
#     'df_readable': df_readable,
#     'df_no_readable': df_no_readable,
#     'df_boring': df_boring,
#     'df_interesting': df_interesting,
#     'df_agenda_yes': df_agenda_yes,
#     'df_text_size_ok': df_text_size_ok,
#     'df_text_size_not_ok': df_text_size_not_ok,
#     'df_positioning_ok': df_positioning_ok,
#     'df_positioning_not_ok': df_positioning_not_ok,
#     'df_pictures': df_pictures,
#     'df_no_pictures': df_no_pictures,
#     'df_tables': df_tables,
#     'df_infographics': df_infographics,
#     'df_no_infographics': df_no_infographics
# }

# # Define the standard columns
# standard_columns = [ 'img_path', 'bullets', 'interesting', 'readable', 'agenda', 'size_of_text', 'tables', 'positioning', 'pictures', 'infographics', 'graphics']

# def has_all_standardize_columns( dataframes, columns):
#     # Add missing columns with NaN values
#     for name, df in dataframes.items():
#         for col in columns:
#             if col not in df.columns:
#                 print("Missing column "+ str(col)+" from dataset "+ str(name))

# has_all_standardize_columns( dataframes, standard_columns)
# data = pd.concat([df_graphics, df_graphics_not, df_bullets, df_no_bullets, df_readable, df_no_readable,df_boring,df_interesting,
#                  df_agenda_yes,df_text_size_ok,df_text_size_not_ok,df_positioning_ok,df_positioning_not_ok,df_pictures,df_no_pictures,
#                  df_tables,df_infographics,df_no_infographics], ignore_index=True)

In [11]:
import pandas as pd

# If 'data' is a list of DataFrames, combine them first
data_df = pd.concat(data, ignore_index=True)

# Define the label columns
label_columns = ['bullets', 'interesting', 'readable', 'agenda', 'size_of_text',
                 'tables', 'positioning', 'pictures', 'infographics', 'graphics']

# Check label counts before balancing
label_counts = data_df[label_columns].sum()
print("Label counts before balancing:")
print(label_counts)

# Balance the dataset (make sure balance_multilabel_df is defined)
df_balanced = balance_multilabel_df(data_df, label_columns, method='undersample')

# Check label counts after balancing
label_counts_balanced = df_balanced[label_columns].sum()
print("Label counts after balancing:")
print(label_counts_balanced)


Label counts before balancing:
bullets          752
interesting      879
readable        1049
agenda           611
size_of_text    1118
tables           462
positioning     1003
pictures         987
infographics     785
graphics         695
dtype: int64
Label counts after balancing:
bullets         640
interesting     699
readable        782
agenda          535
size_of_text    852
tables          389
positioning     674
pictures        695
infographics    625
graphics        512
dtype: object


In [32]:
# #Verify label counts before balancing
# label_columns = ['bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics']
# label_counts = data[label_columns].sum()
# print("Label counts before balancing:")
# print(label_counts)

# # Balance the dataset
# df_balanced = balance_multilabel_df(data, label_columns, method='undersample')

# # Verify the new label counts
# label_counts_balanced = df_balanced[label_columns].sum()
# print("Label counts after balancing:")
# print(label_counts_balanced)

In [12]:
import numpy as np
from sklearn.model_selection import train_test_split

def split_data(images, targets, test_ratio=0.2, val_ratio=0.1):
    """
    Split data into training, validation, and test sets.

    Parameters:
    - images: List or array of images.
    - targets: List or array of targets corresponding to the images.
    - test_ratio: Proportion of the data to include in the test split.
    - val_ratio: Proportion of the training data to include in the validation split.

    Returns:
    - x_train: Training data features.
    - y_train: Training data labels.
    - x_val: Validation data features.
    - y_val: Validation data labels.
    - x_test: Test data features.
    - y_test: Test data labels.
    """
    if not 0 <= test_ratio <= 1:
        raise ValueError("test_ratio must be between 0 and 1.")
    if not 0 <= val_ratio <= 1:
        raise ValueError("val_ratio must be between 0 and 1.")

    # Convert to numpy arrays
    x_data = np.array(images)
    y_data = np.array(targets)

    # Split the data into training+validation and test sets
    x_train_val, x_test, y_train_val, y_test = train_test_split(x_data, y_data, test_size=test_ratio, random_state=42)

    # Calculate the proportion of validation data in the training set
    val_ratio_adjusted = val_ratio / (1 - test_ratio)

    # Split the training+validation data into training and validation sets
    x_train, x_val, y_train, y_val = train_test_split(x_train_val, y_train_val, test_size=val_ratio_adjusted, random_state=42)

    return x_train, y_train, x_val, y_val, x_test, y_test

# Example usage
# images = np.random.random((1000, 64, 64, 3))  # Example data
# targets = np.random.randint(2, size=(1000, 10))  # Example targets

# x_train, y_train, x_val, y_val, x_test, y_test = split_data(images, targets, test_ratio=0.2, val_ratio=0.1)

# print("Training set size:", x_train.shape, y_train.shape)
# print("Validation set size:", x_val.shape, y_val.shape)
# print("Test set size:", x_test.shape, y_test.shape)


In [15]:
# Test loading images
from PIL import Image
def load_img_from_file(path):
    img = Image.open(path)
    img = img.convert('RGB') 
    img = img.resize((224,224))
    return np.array(img)
    # if img != None:
    #     gray = img.convert('L')
    #     return np.array(gray)

def load_images(df):
    images = []
    for index, row in df.iterrows():
        images.append(load_img_from_file(row['img_path']))
    return images


In [16]:
images = load_images(df_balanced)
targets = df_balanced[['bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics']]
x_train, y_train, x_val, y_val, x_test, y_test = split_data(images, targets, test_ratio=0.2, val_ratio=0.1)
print("Training set size:", x_train.shape, y_train.shape)
print("Validation set size:", x_val.shape, y_val.shape)
print("Test set size:", x_test.shape, y_test.shape)

Training set size: (970, 224, 224, 3) (970, 10)
Validation set size: (139, 224, 224, 3) (139, 10)
Test set size: (278, 224, 224, 3) (278, 10)


In [ ]:
# def create_cnn_model(input_shape, num_classes):
#     model = Sequential()

#     # Convolutional Layer 1
#     model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
#     model.add(MaxPooling2D((2, 2)))

#     # Convolutional Layer 2
#     model.add(Conv2D(64, (3, 3), activation='relu'))
#     model.add(MaxPooling2D((2, 2)))

#     # Convolutional Layer 3
#     model.add(Conv2D(128, (3, 3), activation='relu'))
#     model.add(MaxPooling2D((2, 2)))

#     # Flatten the output
#     model.add(Flatten())

#     # Fully Connected Layer 1
#     model.add(Dense(512, activation='relu'))
#     model.add(Dropout(0.5))

#     # Output Layer with num_classes units
#     model.add(Dense(num_classes, activation='sigmoid'))
    
#     print(model.summary())

#     return model


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
#!pip install pydot graphviz


def create_cnn_model(input_shape, num_classes):
    model = Sequential()

    # Convolutional Layer 1
    model.add(Conv2D(32, (3, 3), padding='same', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D((2, 2)))

    # Convolutional Layer 2
    model.add(Conv2D(64, (3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D((2, 2)))

    # Convolutional Layer 3
    model.add(Conv2D(128, (3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D((2, 2)))

    # Convolutional Layer 4
    model.add(Conv2D(256, (3, 3), padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling2D((2, 2)))

    # Flatten the output
    model.add(Flatten())

    # Fully Connected Layer 1
    model.add(Dense(512))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(0.5))

    # Fully Connected Layer 2
    model.add(Dense(256))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(0.5))

    # Output Layer with num_classes units
    model.add(Dense(num_classes, activation='sigmoid'))

    # Compile the model
    optimizer = Adam()
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    # Print the model summary
    print(model.summary())

    # Visualize the model
    #plot_model(model, to_file='oneModel.png', show_shapes=True, show_layer_names=True)

    return model


In [18]:
input_shape = (224, 224, 3)  # Example input shape (64x64 RGB images)
num_classes = 10  # Number of labels

model = create_cnn_model(input_shape, num_classes)


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 224, 224, 32)      896       
                                                                 
 batch_normalization (BatchN  (None, 224, 224, 32)     128       
 ormalization)                                                   
                                                                 
 activation (Activation)     (None, 224, 224, 32)      0         
                                                                 
 max_pooling2d (MaxPooling2D  (None, 112, 112, 32)     0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 112, 112, 64)      18496     
                                                                 
 batch_normalization_1 (Batc  (None, 112, 112, 64)     2

In [20]:
import time
import gc
from tensorflow.keras.callbacks import EarlyStopping
# # Example data (replace with actual data)
# X_train = np.random.random((1000, 64, 64, 3))  # 1000 training samples
# y_train = np.random.randint(2, size=(1000, num_classes))  # Binary labels for each class

# X_val = np.random.random((200, 64, 64, 3))  # 200 validation samples
# y_val = np.random.randint(2, size=(200, num_classes))  # Binary labels for each class

# Ensure the data types are consistent and convert them if necessary
x_train = np.array(x_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int32)
x_val = np.array(x_val, dtype=np.float32)
y_val = np.array(y_val, dtype=np.int32)
start_time = time.time()
# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
model.fit(x_train, y_train, epochs=200, validation_data=(x_val, y_val), callbacks=[early_stopping])
predictions = model.predict(x_test)

end_time = time.time()
training_time = end_time - start_time
print(f"Training time: {training_time:.2f} seconds")

Epoch 1/200
31/31 [==============================] - 7s 82ms/step - loss: 0.7419 - accuracy: 0.1505 - val_loss: 2.6810 - val_accuracy: 0.0072
Epoch 2/200
31/31 [==============================] - 2s 65ms/step - loss: 0.6529 - accuracy: 0.1546 - val_loss: 1.1420 - val_accuracy: 0.0719
Epoch 3/200
31/31 [==============================] - 2s 64ms/step - loss: 0.6007 - accuracy: 0.1629 - val_loss: 0.9104 - val_accuracy: 0.0863
Epoch 4/200
31/31 [==============================] - 2s 62ms/step - loss: 0.5736 - accuracy: 0.1670 - val_loss: 0.9395 - val_accuracy: 0.2446
Epoch 5/200
31/31 [==============================] - 2s 64ms/step - loss: 0.5431 - accuracy: 0.2072 - val_loss: 0.6262 - val_accuracy: 0.2878
Epoch 6/200
31/31 [==============================] - 2s 65ms/step - loss: 0.5057 - accuracy: 0.2289 - val_loss: 0.5863 - val_accuracy: 0.1583
Epoch 7/200
31/31 [==============================] - 2s 62ms/step - loss: 0.4790 - accuracy: 0.2113 - val_loss: 0.6590 - val_accuracy: 0.2518
Epoch 

In [26]:
def ensure_numpy_array(y):
    """
    Ensure the input is a numpy array with the correct type.

    Parameters:
    - y: Input data.

    Returns:
    - Numpy array with the correct type.
    """
    if not isinstance(y, np.ndarray):
        y = np.array(y)
    if y.dtype != np.int32 and y.dtype != np.float32:
        y = y.astype(np.float32)
    return y

In [27]:
from sklearn.metrics import hamming_loss, accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report, jaccard_score, cohen_kappa_score
from sklearn.metrics import hamming_loss, jaccard_score

def evaluate_model(y_pred_probs, y_test):
    """
    Evaluate the model on test data.

    Parameters:
    - y_pred_probs: Predicted probabilities from the model.
    - y_test: True labels for test data.

    Returns:
    - Dictionary of evaluation metrics.
    """
    # Convert predicted probabilities to binary predictions
    y_pred = (y_pred_probs > 0.5).astype(int)

    y_test = ensure_numpy_array(y_test)
    y_pred = ensure_numpy_array(y_pred)
    # Hamming Loss
    h_loss = hamming_loss(y_test, y_pred)

    # Subset Accuracy
    subset_acc = accuracy_score(y_test, y_pred)

    # Precision, Recall, F1-Score with handling zero division
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(y_test, y_pred, average='micro', zero_division=0)

    # ROC AUC Score with handling for labels with only one class present in y_test
    valid_labels = [i for i in range(y_test.shape[1]) if len(np.unique(y_test[:, i])) > 1]
    if valid_labels:
        roc_auc = roc_auc_score(y_test[:, valid_labels], y_pred_probs[:, valid_labels], average='macro')
    else:
        roc_auc = float('nan')  # Not applicable if no valid labels

    # Jaccard Similarity
    jaccard = jaccard_score(y_test, y_pred, average='samples')

    # Kappa Coefficient
    kappa = cohen_kappa_score(y_test.flatten(), y_pred.flatten())

    # Per-label accuracy
    label_accuracies = {}
    for i, label in enumerate(['bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics']):
        label_accuracies[label] = accuracy_score(y_test[:, i], y_pred[:, i])

    # Calculate the average accuracy
    average_accuracy = sum(label_accuracies.values()) / len(label_accuracies)
    print(f"Average Accuracy: {average_accuracy:.4f}")
    
    # Print detailed classification report
    report = classification_report(y_test, y_pred, target_names=[
        'bullets','interesting','readable','agenda','size_of_text','tables','positioning', 'pictures','infographics','graphics'
    ], zero_division=0, output_dict=True)

    # Add accuracy per label to the report
    for label, accuracy in label_accuracies.items():
        if label in report:
            report[label]['accuracy'] = accuracy

    # Print the report
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            print(f"{label}:")
            for metric, value in metrics.items():
                print(f"  {metric}: {value}")
        else:
            print(f"{label}: {metrics}")

    # Summary of evaluation metrics
    metrics = {
        'Hamming Loss': h_loss,
        'Subset Accuracy': subset_acc,
        'Precision (Macro)': precision_macro,
        'Recall (Macro)': recall_macro,
        'F1-Score (Macro)': f1_macro,
        'Precision (Micro)': precision_micro,
        'Recall (Micro)': recall_micro,
        'F1-Score (Micro)': f1_micro,
        'ROC AUC': roc_auc,
        'Jaccard Similarity': jaccard,
        'Kappa Coefficient': kappa,
        'Label Accuracies': label_accuracies
    }

    return metrics

# # Example usage
# # Example test data (replace with actual test data)
# X_test = np.random.random((100, 64, 64, 3))  # 100 test samples
# y_test = np.random.randint(2, size=(100, 10))  # Binary labels for each class
# y_pred_probs = np.random.random((100, 10))  # Predicted probabilities from the model

# Evaluate the model
metrics = evaluate_model(predictions, y_test)
print(metrics)


Average Accuracy: 0.7356
bullets:
  precision: 0.7966101694915254
  recall: 0.7121212121212122
  f1-score: 0.752
  support: 132.0
  accuracy: 0.7769784172661871
interesting:
  precision: 0.7664233576642335
  recall: 0.7191780821917808
  f1-score: 0.7420494699646644
  support: 146.0
  accuracy: 0.737410071942446
readable:
  precision: 0.7093023255813954
  recall: 0.7625
  f1-score: 0.7349397590361446
  support: 160.0
  accuracy: 0.6834532374100719
agenda:
  precision: 0.8194444444444444
  recall: 0.6020408163265306
  f1-score: 0.6941176470588235
  support: 98.0
  accuracy: 0.8129496402877698
size_of_text:
  precision: 0.7058823529411765
  recall: 0.8727272727272727
  f1-score: 0.7804878048780488
  support: 165.0
  accuracy: 0.7086330935251799
tables:
  precision: 0.775
  recall: 0.40789473684210525
  f1-score: 0.5344827586206896
  support: 76.0
  accuracy: 0.8057553956834532
positioning:
  precision: 0.6992481203007519
  recall: 0.6788321167883211
  f1-score: 0.6888888888888889
  suppor